# Chapter 6 — Your Model Is a Dependency

**Book alignment:** DSPy From First Principles, Chapter 6

**Question this notebook isolates:** Can a run prove which LM dependency and configuration executed it — while keeping fallback availability distinct from model evidence?


In [ ]:
from pathlib import Path
import sys


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "experiments" / "dspy-from-first-principles" / "common").exists():
            return candidate
    raise RuntimeError(
        "Run this notebook from a checkout containing experiments/dspy-from-first-principles"
    )


REPO_ROOT = find_repo_root(Path.cwd().resolve())
EXP_ROOT = REPO_ROOT / "experiments" / "dspy-from-first-principles"
sys.path.insert(0, str(EXP_ROOT))

from common.data import teaching_cases
from common.fingerprints import fingerprint
from common.provider import CANONICAL_PROVIDER, canonical_dspy_lm_kwargs


## Configuration identity is hashable

The friendly model name is not the dependency; the seven-field configuration is. Mutating one field at a time — without any model call — must change the fingerprint, or two different experiments could claim to be the same one.


In [ ]:
canonical = canonical_dspy_lm_kwargs()
base_fp = fingerprint(canonical)

def mutated(**overrides):
    cfg = dict(canonical)
    cfg.update(overrides)
    return cfg

MUTATIONS = {
    "model_changed": mutated(model="ollama_chat/identity-test-model"),
    "endpoint_changed": mutated(api_base="http://127.0.0.1:11435"),
    "temperature_changed": mutated(temperature=0.25),
    "think_changed": mutated(think=True),
    "max_tokens_changed": mutated(max_tokens=int(canonical["max_tokens"]) + 64),
}
for name, cfg in MUTATIONS.items():
    print(f"{name:20s} fingerprint={fingerprint(cfg)[:12]}... changed={fingerprint(cfg) != base_fp}")
print(f"canonical fingerprint={base_fp[:12]}...")


In [ ]:
assert all(fingerprint(cfg) != base_fp for cfg in MUTATIONS.values())
assert len({fingerprint(cfg) for cfg in MUTATIONS.values()}) == len(MUTATIONS)
print("'we used Qwen3 both times' is not an experimental identity; the hash is")


## Silent fallback corrupts evidence

A bare `except: return the original sentence` keeps the interface alive and poisons every downstream claim — an outage scores as a mediocre day. The wrapper below mirrors the experiment runner: degraded output is usable, but explicitly ineligible as model evidence.


In [ ]:
class InjectedDependencyFailure:
    """Deterministic dependency failure; makes no provider call."""

    def __call__(self, **inputs):
        raise RuntimeError("injected LM dependency failure")


def run_with_explicit_fallback(callable_program, inputs, model_name: str) -> dict:
    try:
        pred = callable_program(**inputs)
        return {
            "output": pred,
            "used_real_lm": True,
            "eligible_as_model_evidence": True,
            "fallback_state": None,
            "error": None,
        }
    except Exception as exc:
        return {
            "output": {
                "rewritten_text": inputs["sentence"],
                "rationale": "Fallback preserved the original sentence.",
            },
            "used_real_lm": False,
            "eligible_as_model_evidence": False,
            "fallback_state": "lm_unavailable",
            "error": f"{type(exc).__name__}: {exc}",
        }


ed001 = {c.case_id: c for c in teaching_cases()}["ed-001"]
measured_inputs = {"sentence": ed001.sentence, "goal": ed001.goal, "context": ed001.context}
fallback = run_with_explicit_fallback(InjectedDependencyFailure(), measured_inputs, CANONICAL_PROVIDER.model)
print(fallback)


In [ ]:
assert fallback["output"]
assert fallback["used_real_lm"] is False
assert fallback["eligible_as_model_evidence"] is False
assert fallback["fallback_state"] == "lm_unavailable"
assert fallback["output"]["rewritten_text"] == ed001.sentence
print("output exists; no claim is made that the model produced it")


## Provenance explains; it does not guarantee

Temperature zero removes one intentional source of sampling randomness — not cross-session drift or order effects. Two recorded ed-033-style alternatives share an identical input fingerprint and configuration fingerprint yet diverge in output, which is exactly why run records must be kept.


In [ ]:
ALT_A = "We have made a good, honest cup of coffee since 1998."
ALT_B = "We have brewed an honest cup of coffee every day since 1998."
input_fp = fingerprint(measured_inputs)
fp_a = fingerprint({"rewritten_text": ALT_A})
fp_b = fingerprint({"rewritten_text": ALT_B})
record = {
    "program": "editorial_rewrite_program",
    "lm_model": CANONICAL_PROVIDER.model,
    "used_real_lm": True,
    "input_fingerprint": input_fp,
    "config_fingerprint": base_fp,
}
print(f"input fingerprint:  {input_fp[:12]}...")
print(f"output A fingerprint: {fp_a[:12]}... ({ALT_A})")
print(f"output B fingerprint: {fp_b[:12]}... ({ALT_B})")


In [ ]:
assert fp_a != fp_b
assert record["input_fingerprint"] == input_fp
assert record["config_fingerprint"] == base_fp
print("same recorded config, different outputs: temperature=0 is not determinism")


## What we earned

The model is now a dependency with a hashable identity, a failure mode distinguishable from a bad result, and a role recorded beside the program. Small deltas from one execution schedule are not self-interpreting — from here on they need paired repetition and, where available, a zero-effect negative control.

Notebook 07 / Chapter 7 fixes the other missing half: we can specify, execute, and record a program, but we still have no cases we trust and no criterion worth optimizing. Examples are data, not decoration.
